### Web Information Extraction and Retrieval (WIER)  
# RoadRunner Algorithm: Explanation and Implementation

This notebook presents the RoadRunner algorithm, a method for automatic wrapper induction from web pages with similar templates. It is commonly used in Web Information Extraction to derive structured data from semi-structured HTML documents.

We will:

- Explain the concept and motivation behind RoadRunner.
- Parse HTML pages into simplified DOM trees.
- Implement the core alignment algorithm to generalize a wrapper.
- Demonstrate how to use the wrapper to extract data from new documents.


## Concept: What is RoadRunner?

The RoadRunner algorithm is a template-independent wrapper induction technique. It compares the structure of multiple HTML documents that share a common layout (e.g., product listings, news articles) and identifies the parts that vary.

By analyzing these differences and similarities, RoadRunner generates a wrapper, a kind of generalized tree structure. This wrapper can later be used to extract content (data fields) from similar web pages.

RoadRunner generalizes the HTML DOM structure using:

- `#PCDATA` to indicate text content.
- `?` to denote optional elements.
- `+` to denote repeated elements or patterns (not implemented in full here).

This notebook walks through a simplified version of RoadRunner to illustrate its basic ideas.

<img src="https://i.ibb.co/PGV9wt0P/roadrunner.png" alt="Image of the roadrunner algorithm" border="0">


## Step 1: Parse HTML Pages to DOM Trees

We begin by parsing two HTML documents into a simplified DOM representation. We will use the BeautifulSoup library to process the HTML content. Each HTML element will be represented as a node with a tag and children (which can be other nodes or text).

This transformation allows us to easily compare and align the structure of the web pages.


In [ ]:
from bs4 import BeautifulSoup
from typing import List, Union

class Node:
    def __init__(self, tag: str, children: List[Union['Node', str]] = None, repeating: bool = False):
        self.tag = tag
        self.children = [child for child in children if isinstance(child, Node) or len(child) > 0] if children else []
        self.repeating = repeating
        self.optional = False

    def __repr__(self, level=0):
        indent = "  " * level
        if isinstance(self, str):
            return indent + self  # Indent text nodes

        # Start with the opening tag and a newline
        if self.repeating or self.optional:
            rep = indent + f"(<{self.tag}>\n"
        else:
            rep = indent + f"<{self.tag}>\n"

        # Add child nodes' representations with increased indentation
        for child in self.children:
            if isinstance(child, Node):
                rep += child.__repr__(level + 1) + "\n" # Recursive call with indent + 1
            else:
                rep += indent + "  " + child + "\n" # Indent text content further

        # Close the tag with the original indentation
        if self.repeating:
            rep += indent + f"</{self.tag}>)+"
        elif self.optional:
            rep += indent + f"</{self.tag}>)?"
        else:
            rep += indent + f"</{self.tag}>"
        return rep

def html_to_tree(html: str) -> Node:
    soup = BeautifulSoup(html, 'html.parser')

    def build(node) -> Union[Node, str]:
        if node.name is None:
            return node.string.strip()
        children = [build(child) for child in node.children]
        return Node(node.name, children)

    return build(soup.body or soup)

### Example: Parsing Sample HTML Pages

Here we provide two example HTML pages with a similar structure and different content. We will parse each into a DOM-like tree using the function defined above.


In [ ]:
html1 = """
<html><body><div><h1>Title A</h1><p>Item 1</p><p>Item 2</p></div></body></html>
"""

html2 = """
<html><body><div><h1>Title B</h1><p>Item 3</p><p>Item 4</p></div></body></html>
"""

tree1 = html_to_tree(html1)
tree2 = html_to_tree(html2)

print("Tree 1:\n", tree1)
print("\nTree 2:\n", tree2)


## Step 2: Tree Alignment (Generalization)

We now align the DOM trees generated in the previous step. The alignment process identifies matching tags and generalizes differing content using placeholders.

This function compares nodes from two trees:
- If tags match, their children are recursively aligned.
- If text content differs, it is replaced by `#PCDATA`.
- If an element exists only in one tree, it is marked as optional using `?`.


In [ ]:
def align_nodes(n1: Union[Node, str], n2: Union[Node, str]) -> Union[Node, str]:
    if isinstance(n1, str) or isinstance(n2, str):
        return "#PCDATA"

    if n1.tag != n2.tag:
        return "#PCDATA"  # Different tags can't be aligned in this simplified version

    aligned_children = []
    len1, len2 = len(n1.children), len(n2.children)
    i = j = 0

    while i < len1 and j < len2:
        child1, child2 = n1.children[i], n2.children[j]
        aligned = align_nodes(child1, child2)
        aligned_children.append(aligned)
        i += 1
        j += 1

    # Handle extra nodes (optional)
    for k in range(i, len1):
        aligned_children.append("?")
    for k in range(j, len2):
        aligned_children.append("?")

    return Node(n1.tag, aligned_children)


### Example: Generalizing Two Trees

We now align the two DOM trees and output the generalized structure that serves as the learned wrapper.


In [ ]:
generalized_tree = align_nodes(tree1, tree2)
print("Generalized Tree:\n", generalized_tree)


## Step 3: Handle Repetitions

In real-world web pages, data often appears in repeated structures (e.g., product listings, news entries). RoadRunner handles such cases using the `+` operator to denote repetition. Elements that are present in some pages but missing in others are marked as optional using `?`.

In this enhanced implementation, we:
- Detect repetitions by checking for multiple similar subtrees.
- Align and annotate repeated segments using the `+` suffix.
- Mark elements that appear in one structure but not in the other as optional (`?`).

These generalizations help create a flexible wrapper for data extraction.

In [ ]:
def node_structures_match(n1, n2):
    if isinstance(n1, Node) and isinstance(n2, Node):
        if n1.tag != n2.tag:
            return False
        return all(node_structures_match(c1, c2) for c1, c2 in zip(n1.children, n2.children))
    return isinstance(n1, str) and isinstance(n2, str)

def find_repeating_children(node):
    i = 1
    while i < len(node.children):
        if node_structures_match(node.children[i-1], node.children[i]):
            node.children[i-1].repeating = True
            node.children.pop(i)
        else:
            i += 1


In [ ]:
def nodes_are_similar(n1, n2):
    if isinstance(n1, Node) and isinstance(n2, Node):
        return n1.tag == n2.tag
    return isinstance(n1, str) and isinstance(n2, str)

def align_nodes_enhanced(n1: Union[Node, str], n2: Union[Node, str]) -> Union[Node, str]:
    if isinstance(n1, str) or isinstance(n2, str):
        if n1 == n2:
            return n1
        else:
            return "#PCDATA"

    if n1.tag != n2.tag:
        return "#PCDATA"

    find_repeating_children(n1)
    find_repeating_children(n2)

    aligned_children = []
    children1 = n1.children
    children2 = n2.children
    i = j = 0

    while i < len(children1) and j < len(children2):
        c1 = children1[i]
        c2 = children2[j]

        if nodes_are_similar(c1, c2):
            aligned = align_nodes_enhanced(c1, c2)
            aligned_children.append(aligned)
            i += 1
            j += 1
        elif (i + 1 < len(children1) and nodes_are_similar(children1[i + 1], c2)):
            c1.optional = True
            aligned_children.append(c1)
            i += 1
        elif (j + 1 < len(children2) and nodes_are_similar(c1, children2[j + 1])):
            c2.optional = True
            aligned_children.append(c2)
            j += 1
        else:
            aligned_children.append(align_nodes_enhanced(c1, c2))
            i += 1
            j += 1

    while i < len(children1):
        aligned_children.append("?")
        i += 1
    while j < len(children2):
        aligned_children.append("?")
        j += 1

    return Node(n1.tag, aligned_children, repeating=n1.repeating or n2.repeating)


## Example: Aligning Pages with Repetition and Optional Elements

The following HTML samples have:
- Titles that change.
- A list of `<p>` items which can vary in length and content.
- An optional `<img>` in one page.

We will align them and show the resulting wrapper.


In [ ]:
html1 = """
<html><body>
<div>
    <h1>Title A</h1>
    <img src="a.jpg"/>
    <p>Item 1</p>
    <p>Item 2</p>
</div></body></html>
"""

html2 = """
<html><body>
<div>
    <h1>Title B</h1>
    <p>Item 3</p>
    <p>Item 4</p>
    <p>Item 5</p>
</div></body></html>
"""


tree1 = html_to_tree(html1)
tree2 = html_to_tree(html2)
wrapper_tree = align_nodes_enhanced(tree1, tree2)
print("Generated Wrapper:\n", wrapper_tree)


## Step 4: Extract Data from New Pages

Using the generalized wrapper, we can extract the variable content from new web pages that follow the same template. The function below navigates through the new tree and collects content from nodes marked `#PCDATA` in the wrapper.


In [ ]:
def extract_data(node: Union[Node, str], template: Union[Node, str]) -> List[str]:
    if template == "#PCDATA":
        return [node] if isinstance(node, str) else []

    if isinstance(node, str) or node.tag != template.tag:
        return []

    data = []
    template_i = 0
    node_i = 0
    while node_i < len(node.children) and template_i < len(template.children):
        child_node = node.children[node_i]
        child_template = template.children[template_i]
        if isinstance(child_node, str) and isinstance(child_template, str):
            data.extend(extract_data(child_node, child_template))
            template_i += 1
            node_i += 1
        elif isinstance(child_node, str) and isinstance(child_template, str):
            template_i += 1
            node_i += 1
        elif child_template.repeating and child_node.tag == child_template.tag:
            data.extend(extract_data(child_node, child_template))
            node_i += 1
        elif child_template.optional and child_node.tag != child_template.tag:
            template_i += 1
        else:
            data.extend(extract_data(child_node, child_template))
            template_i += 1
            node_i += 1

    return data


### Example: Data Extraction

We apply the wrapper to a new HTML document to extract its contents.


In [ ]:
new_html = """
<html><body>
<div>
    <h1>Title C</h1>
    <p>Item 6</p>
    <p>Item 7</p>
    <p>Item 8</p>
    <p>Item 9</p>
</div></body></html>
"""
new_tree = html_to_tree(new_html)
extracted = extract_data(new_tree, wrapper_tree)
print("Extracted Data:", extracted)


## Summary

In this notebook, we implemented a simplified version of the RoadRunner algorithm. The key steps included:

1. Parsing HTML into a tree structure.
2. Aligning multiple trees to generate a wrapper using generalization.
3. Applying the wrapper to extract content from new pages.

Future extensions could include handling:
- Repeated elements using `+` operator.
- Alignment of sequences using dynamic programming.
- Robust error handling and tag normalization.

This serves as a foundational demonstration for the Web Information Extraction and Retrieval course.
